<a href="https://colab.research.google.com/github/polazzoza/INFO648/blob/main/INFO648/Lesson7HW/Predicting_B2B_SaaS_Deal_Value.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Predicting B2B SaaS Deal Value

Here is me uploading pandas and scikit

In [21]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

#Part 1 [a] - Lo[a]d and explore

In [3]:
#1

#Here is me uploading the data
df=pd.read_csv("/content/saas_deals.csv")

In [6]:
#2
df.shape


(300, 8)

In [7]:
df.describe(include='all')


,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region,acv_usd
count,300,300.000000,300.000000,300.000000,300,300,300,300.000000
unique,300,NaN,NaN,NaN,5,3,3,NaN
top,D-0173,NaN,NaN,NaN,Tech,Pro,NorthAmerica,NaN
freq,1,NaN,NaN,NaN,80,144,170,NaN
mean,NaN,1177.373333,15.923333,92.843333,NaN,NaN,NaN,69109.313333
std,NaN,1717.906689,8.889368,48.290566,NaN,NaN,NaN,34415.112768
min,NaN,16.000000,1.000000,5.000000,NaN,NaN,NaN,5000.000000
25%,NaN,62.000000,8.000000,55.500000,NaN,NaN,NaN,43925.500000
50%,NaN,303.500000,17.000000,93.000000,NaN,NaN,NaN,60715.500000
75%,NaN,1675.250000,24.000000,130.500000,NaN,NaN,NaN,90609.750000


3.
The numeric collumns here are 'company_employees','sales_calls','pipeline_days', and "acv_usd."

The categorical collumns here are 'industry', 'plan_tier', 'region.'

We are dropping deal_id

Since the goal of the project is a model that can predict ACV from our features, everything else here, our target in this case is 'acv_used'

#Part 2 [b] - Train / test split

In [16]:
#4
#Making the variables and splitting the data into test and train

X = df[['company_employees','sales_calls','pipeline_days', 'industry', 'plan_tier', 'region']] #Our features
y = df['acv_usd'] #Our target

#Split the data into test and train
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=0)

In [17]:
#How many rows in each set
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(225, 6)
(75, 6)
(225,)
(75,)


5. The reason that we set aside test data is because we "don't want to give the model all the answers" (like mentioned in class). We want to see how accurate the model we made is using the data that we have on hand, and the way to do that is to withhold that test data and letting it try to figure out what happened itself.

#Part 3 [c] - Build the Pipleing

In [26]:
#7

#Categorizing what is a numeric collumn and what is a categorical collumn

numeric_cols =['company_employees','sales_calls','pipeline_days']
categorical_cols =['industry', 'plan_tier', 'region']



#Making the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',
         OneHotEncoder(drop='first',handle_unknown='ignore'),
         categorical_cols),
        ('num', 'passthrough', numeric_cols),
    ]
)

In [27]:
#8
#Wrapping the ColumnTransformer and LinearRegression() in one pipleline

model = Pipeline([
    ('prep', preprocessor),
    ('lr', LinearRegression() )

  ]

)

model

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['industry', 'plan_tier',
                                                   'region']),
                                                 ('num', 'passthrough',
                                                  ['company_employees',
                                                   'sales_calls',
                                                   'pipeline_days'])])),
                ('lr', LinearRegression())])

#Part 4 [d] Fit and evaluate

In [28]:
#9
#fitting the training data
model.fit(X_train, y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['industry', 'plan_tier',
                                                   'region']),
                                                 ('num', 'passthrough',
                                                  ['company_employees',
                                                   'sales_calls',
                                                   'pipeline_days'])])),
                ('lr', LinearRegression())])

In [29]:
#10
#Model Prediction vs Actual Data
y_pred=model.predict(X_test)
preview=pd.DataFrame({'actual':y_test,'Predict':y_pred})
preview.head()

,actual,Predict
208,50509,45152.670831
188,41939,32543.782317
12,106150,120961.820871
221,52338,61499.723921
239,98709,79724.898795


In [32]:
#11
# Here is the Mean Absolute Error (MAE) and RMSE (Root Mean Squared Error)
mae=mean_absolute_error(y_test,y_pred)
rmse=root_mean_squared_error(y_test,y_pred)
print(f':MAE =${mae} and RMSE=${rmse}')


:MAE =$10626.682441190758 and RMSE=$13444.1121105072


MAE =$10,626.682441190758

and

RMSE=$13,444.1121105072


12.
On average, according to our MAE, the model is off the actual on average around $10,626.682441190758,

while using RSME, it is off by about
$13,444.1121105072

Part 5 [e] - Int[e]pret the coefficients

In [38]:
#13
feature_names = model.named_steps['prep'].get_feature_names_out()
coefs = model.named_steps['lr'].coef_
intercept = model.named_steps['lr'].intercept_

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs.round(2)
})
print(f'intercept = ${intercept:,.2f}')
coef_table

intercept = $35,803.08


,feature,coefficient
0,cat__industry_Healthcare,-2364.04
1,cat__industry_Manufacturing,-16830.97
2,cat__industry_Retail,-23918.55
3,cat__industry_Tech,-19276.81
4,cat__plan_tier_Enterprise,80206.88
5,cat__plan_tier_Pro,18461.95
6,cat__region_EMEA,4497.90
7,cat__region_NorthAmerica,-5801.32
8,num__company_employees,5.17
9,num__sales_calls,666.51


14.

 Dropped Basiline

In [ ]:
#Here are all the unique values in each category collumn

In [43]:
print("Region Unique Values: ", df['region'].unique())
print("Industry Unique Values: ", df['industry'].unique())
print("Plan Tier Unique Values: ", df['plan_tier'].unique())

Region Unique Values:  ['APAC' 'EMEA' 'NorthAmerica']
Industry Unique Values:  ['Retail' 'Healthcare' 'Manufacturing' 'Tech' 'Finance']
Plan Tier Unique Values:  ['Enterprise' 'Pro' 'Basic']


Looking at the printing unique values and comparing it against the coef chart, we see the dropped region value is APAC, the dropped Industry value is Finance, and the dropped Plan tier is Basic

15.
  Intrepretation of Coef:

Employees: For every additional employee, the ACV would increase by $5.17, all else staying the same.

EMEA Region: For every addtional contract created in the EMEA, the ACV would be expected to increase by $4497.90, all else standing.

Sales Calls: For every additional sales call, the ACV would increase bu $666.51, all elsee holding constant.

